# 比特频谱扫描

一次调用只执行给定频率范围和步进的一次扫描。实验结果自动写入 `output/experiments/`，Web 控制台负责查看结果。

In [5]:
from uuid import uuid4

from sqvm.calibration import (
    apply_calibration_candidates_to_current_configuration,
    run_spectroscopy,
)

print('SQVM 用户接口导入成功')

SQVM 用户接口导入成功


## 扫描参数

每个对象填写 `(起始频率, 终止频率)`，单位为 GHz。一个对象执行单比特扫描，两个对象自动并行；并行时两个范围必须按公共步进产生相同数量的数据点。脉冲和设备参数使用 API 默认值。

In [6]:
FREQUENCY_RANGES_GHZ = {
    'Q1': (5.00, 5.40),
    'Q2': (5.10, 5.50),
}
FREQUENCY_STEP_GHZ = 0.02
OPERATION_ID = str(uuid4())
RUN_EXPERIMENT = True
UPDATE_PARAMETERS = False

In [7]:
def report_progress(event):
    action = '开始' if event['event'] == 'circuit_started' else '完成'
    current = event['completed'] + 1 if event['event'] == 'circuit_started' else event['completed']
    print(f"{action} {current}/{event['total']}: {event['circuit_id']}")

if RUN_EXPERIMENT:
    result = run_spectroscopy(
        FREQUENCY_RANGES_GHZ,
        frequency_step_GHz=FREQUENCY_STEP_GHZ,
        operation_id=OPERATION_ID,
        progress_callback=report_progress,
    )
    print(f'运行 ID: {result.run_id}')
    print(f'结果目录: {result.root}')
    for target, data in result.data.items():
        print(f'{target} frequency_GHz: {list(data["frequency_GHz"])}')
        print(f'{target} P1: {list(data["P1"])}')
        print(f'{target} peak: {result.analysis.peaks[target]}')
        print(f'{target} candidate: {result.candidates[target]}')
else:
    print('参数已设置。将 RUN_EXPERIMENT 改为 True 后重新运行本单元格。')

开始 1/21: sp_379456d1777235372c23025c8280b17e
完成 1/21: sp_379456d1777235372c23025c8280b17e
开始 2/21: sp_9bc6bb236a6e8425fef6451a5af5a4d8
完成 2/21: sp_9bc6bb236a6e8425fef6451a5af5a4d8
开始 3/21: sp_74b200c207147516c0a56c5cc080954a
完成 3/21: sp_74b200c207147516c0a56c5cc080954a
开始 4/21: sp_506236d600310aca18ce500bd980216f
完成 4/21: sp_506236d600310aca18ce500bd980216f
开始 5/21: sp_760135741e66a37194067251806f498a
完成 5/21: sp_760135741e66a37194067251806f498a
开始 6/21: sp_368a02e96886fa4aadacb0ba514627cf
完成 6/21: sp_368a02e96886fa4aadacb0ba514627cf
开始 7/21: sp_e0923eff8acc77b5f7a91ca9ed238f14
完成 7/21: sp_e0923eff8acc77b5f7a91ca9ed238f14
开始 8/21: sp_757a5179e23bd0b1b53230de0f059349
完成 8/21: sp_757a5179e23bd0b1b53230de0f059349
开始 9/21: sp_797f87fe965c92d26914ab14797e8c88
完成 9/21: sp_797f87fe965c92d26914ab14797e8c88
开始 10/21: sp_6c56bab415145a78be1ce7dca42d913b
完成 10/21: sp_6c56bab415145a78be1ce7dca42d913b
开始 11/21: sp_53288cd328848c29827d7d04c0e2b4ef
完成 11/21: sp_53288cd328848c29827d7d04c0e2b4ef
开始 12/

## 候选校准值

本次扫描的有效峰值会形成候选频率，但不会自动修改配置。检查候选、对比度和门限后，将 `UPDATE_PARAMETERS` 改为 `True`，再运行下方单元格。

In [8]:
if UPDATE_PARAMETERS:
    if not RUN_EXPERIMENT or 'result' not in globals():
        raise RuntimeError('请先运行实验并检查候选校准值')
    confirmation = f'APPLY CALIBRATION CANDIDATES {result.run_id}'
    update = apply_calibration_candidates_to_current_configuration(
        result,
        confirmation_phrase=confirmation,
    )
    print(f'已更新当前配置: {update.device_id} r{update.current_revision}')
    print(f'更新目标: {update.targets}')
    print(f'已更新参数: {dict(update.applied_values)}')
else:
    print('候选值尚未写入当前配置；确认后将 UPDATE_PARAMETERS 改为 True。')

候选值尚未写入当前配置；确认后将 UPDATE_PARAMETERS 改为 True。


## 查看结果

启动 `start_calibration_web.cmd` 后访问 [http://127.0.0.1:8765/#/experiments](http://127.0.0.1:8765/#/experiments)。